# Recipe Ingredients — Extraction & Cleaning

Extraction et nettoyage de la liste d'ingrédients depuis le dataset de 2,2M recettes.

**Colonnes du dataset :**
- `title` : nom de la recette
- `ingredients` : liste brute avec quantités
- `NER` : ingrédients extraits par NER (sans quantités) ← on utilise ça
- `directions`, `link`, `source`, `site`

## 📚 Import Libraries

In [1]:
import ast
import re
from collections import Counter

import pandas as pd

## 📂 Chargement des données

In [2]:
data = pd.read_csv("raw_data/recipes_data.csv")

In [3]:
print(f"Shape     : {data.shape}")
print(f"Colonnes  : {list(data.columns)}")
print(f"Valeurs manquantes :\n{data.isnull().sum()}\n")
data.head(5)

Shape     : (2231142, 7)
Colonnes  : ['title', 'ingredients', 'directions', 'link', 'source', 'NER', 'site']
Valeurs manquantes :
title          1
ingredients    0
directions     0
link           0
source         0
NER            0
site           0
dtype: int64



,title,ingredients,directions,link,source,NER,site
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""bite size shredded rice biscuits"", ""vanilla""...",www.cookbooks.com
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""cream of mushroom soup"", ""beef"", ""sour cream...",www.cookbooks.com
2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""pepper"", ""cream cheese"", ""gar...",www.cookbooks.com
3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[""chicken gravy"", ""cream of mushroom soup"", ""c...",www.cookbooks.com
4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[""graham cracker crumbs"", ""powdered sugar"", ""p...",www.cookbooks.com


## 🧹 Extraction & Nettoyage des ingrédients

In [4]:
KNOWN_BRANDS = {
    "cool whip", "velveeta", "crisco", "jello", "jell-o", "kraft",
    "bisquick", "campbells", "campbell", "heinz", "hellmanns", "hellmann",
    "karo", "lipton", "miracle whip", "pillsbury", "realemon", "tabasco",
    "worcestershire", "spam", "oreo", "ritz", "swanson", "rotel", "ro-tel",
    "duncan hines", "betty crocker", "knorr", "maggi", "frenchs",
    "hidden valley", "pace", "bushs", "chef boyardee", "progresso",
    "del monte", "green giant", "quaker", "philadelphia",
    "dorito", "doritos", "vegenaise", "squirt", "campbells golden",
}

NON_FOOD = {
    "grill pan", "cookie cutters", "cookie cutter", "cooking spray",
    "wet ingredients", "dry ingredients", "kind deeds", "medley",
    "pan spray", "nonstick spray",
}

def parse_ner(row):
    try:
        return ast.literal_eval(row)
    except (ValueError, SyntaxError):
        return []

def clean_ingredient(ingredient):
    if not ingredient or pd.isna(ingredient):
        return None
    cleaned = ingredient.strip().lower()
    cleaned = re.sub(r"[^a-z\s\-]", "", cleaned)
    cleaned = re.sub(r"^(a|an|the|some)\s+", "", cleaned).strip()
    if len(cleaned) <= 2:
        return None
    return cleaned

def is_valid(ing):
    if ing in KNOWN_BRANDS or ing in NON_FOOD:
        return False
    if re.search(r'\b(de|of|with|and|or|a|the|in|for)$', ing):
        return False
    if ing.startswith("-") or ing.endswith("-"):
        return False
    return True

# Passe 1 : comptage des fréquences
MIN_FREQUENCY = 200
ingredient_counter = Counter()
for row in data["NER"]:
    for ing in parse_ner(row):
        cleaned = clean_ingredient(ing)
        if cleaned:
            ingredient_counter[cleaned] += 1

# Liste finale des ingrédients valides
valid_ingredients = {
    ing for ing, n in ingredient_counter.items()
    if n >= MIN_FREQUENCY and is_valid(ing)
}

print(f"Ingrédients bruts          : {len(ingredient_counter):>10,}")
print(f"Ingrédients valides (≥{MIN_FREQUENCY}) : {len(valid_ingredients):>10,}")

Ingrédients bruts          :    194,914
Ingrédients valides (≥200) :      3,114


## 💾 Nettoyage final du dataset

In [5]:
# Passe 2 : enrichissement du dataset
def get_clean_ingredients(row):
    return [
        ing for raw in parse_ner(row)
        if (ing := clean_ingredient(raw)) and ing in valid_ingredients
    ]

data["clean_ingredients"] = data["NER"].apply(get_clean_ingredients)

avg = data["clean_ingredients"].apply(len).mean()
empty = (data["clean_ingredients"].apply(len) == 0).sum()
print(f"Moyenne d'ingrédients par recette : {avg:.1f}")
print(f"Recettes sans ingrédient valide   : {empty:,}")
data[["title", "NER", "clean_ingredients"]].head(5)

Moyenne d'ingrédients par recette : 7.8
Recettes sans ingrédient valide   : 4,543


,title,NER,clean_ingredients
0,No-Bake Nut Cookies,"[""bite size shredded rice biscuits"", ""vanilla""...","[vanilla, brown sugar, nuts, milk, butter]"
1,Jewell Ball'S Chicken,"[""cream of mushroom soup"", ""beef"", ""sour cream...","[cream of mushroom soup, beef, sour cream, chi..."
2,Creamy Corn,"[""frozen corn"", ""pepper"", ""cream cheese"", ""gar...","[frozen corn, pepper, cream cheese, garlic pow..."
3,Chicken Funny,"[""chicken gravy"", ""cream of mushroom soup"", ""c...","[chicken gravy, cream of mushroom soup, chicke..."
4,Reeses Cups(Candy),"[""graham cracker crumbs"", ""powdered sugar"", ""p...","[graham cracker crumbs, powdered sugar, peanut..."


In [6]:
before = len(data)

data = data[data["clean_ingredients"].apply(len) > 0].reset_index(drop=True)

print(f"Recettes avant   : {before:,}")
print(f"Supprimées       : {before - len(data):,}")
print(f"Recettes finales : {len(data):,}")
data[["title", "NER", "clean_ingredients"]].head(5)

Recettes avant   : 2,231,142
Supprimées       : 4,543
Recettes finales : 2,226,599


,title,NER,clean_ingredients
0,No-Bake Nut Cookies,"[""bite size shredded rice biscuits"", ""vanilla""...","[vanilla, brown sugar, nuts, milk, butter]"
1,Jewell Ball'S Chicken,"[""cream of mushroom soup"", ""beef"", ""sour cream...","[cream of mushroom soup, beef, sour cream, chi..."
2,Creamy Corn,"[""frozen corn"", ""pepper"", ""cream cheese"", ""gar...","[frozen corn, pepper, cream cheese, garlic pow..."
3,Chicken Funny,"[""chicken gravy"", ""cream of mushroom soup"", ""c...","[chicken gravy, cream of mushroom soup, chicke..."
4,Reeses Cups(Candy),"[""graham cracker crumbs"", ""powdered sugar"", ""p...","[graham cracker crumbs, powdered sugar, peanut..."


In [9]:
data[["NER", "clean_ingredients"]]


,NER,clean_ingredients
0,"[""bite size shredded rice biscuits"", ""vanilla""...","[vanilla, brown sugar, nuts, milk, butter]"
1,"[""cream of mushroom soup"", ""beef"", ""sour cream...","[cream of mushroom soup, beef, sour cream, chi..."
2,"[""frozen corn"", ""pepper"", ""cream cheese"", ""gar...","[frozen corn, pepper, cream cheese, garlic pow..."
3,"[""chicken gravy"", ""cream of mushroom soup"", ""c...","[chicken gravy, cream of mushroom soup, chicke..."
4,"[""graham cracker crumbs"", ""powdered sugar"", ""p...","[graham cracker crumbs, powdered sugar, peanut..."
...,...,...
2226594,"[""chocolate hazelnut spread"", ""marshmallows"", ...","[marshmallows, hazelnuts, tortillas, butter]"
2226595,"[""choice"", ""miracle whip"", ""eggs"", ""relish"", ""...","[choice, eggs, relish, paprika, salt]"
2226596,"[""soy sauce"", ""radish"", ""white sesame seeds"", ...","[soy sauce, radish, white sesame seeds, sesame..."
2226597,"[""apple cider"", ""egg"", ""sugar"", ""freshly groun...","[apple cider, egg, sugar, freshly ground black..."
